In [ ]:
from pathlib import Path

from astropy import table
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

from ugdatalab import GaiaData
from ugdatalab.methods.bayesian.likelihoods import LinearGaussianLikelihood
from ugdatalab.methods.bayesian.mcmc import nuts_sample

from plotters import (
    plot_aitoff_reddening,
    plot_reddening_quality_diagnostics,
    plot_sfd_comparison,
    plot_sfd_regime_decomposition,
)

R_G = 2.0
MIN_BP_SNR = 5.0
MIN_RP_SNR = 5.0
MAX_SIGMA_E = 0.15
MIN_EBPRP = 0.0
MAX_EBPRP = 10.0

In [ ]:
def build_pc_arrays(data):
    """Return (x_centered, bp_rp, sigma_color) for a period-color fit."""
    log_p = np.log10(data["rrlyrae_representative_period"])
    bp_rp = np.array(data["bp_rp"])
    snr_bp = np.array(data["phot_bp_mean_flux_over_error"])
    snr_rp = np.array(data["phot_rp_mean_flux_over_error"])
    sigma_color = (2.5 / np.log(10)) * np.sqrt(1 / snr_bp**2 + 1 / snr_rp**2)
    return log_p - np.mean(log_p), bp_rp, sigma_color


def compute_extinction(data, pc_result, mean_log_p, r_g=R_G):
    """Compute empirical E(BP-RP) and A_G for each star.

    Returns the table with added columns: E_bprp, A_G_calc, sigma_E.
    """
    out = data.copy()
    log_p = np.log10(np.array(data["rrlyrae_representative_period"]))
    bp_rp_obs = np.array(data["bp_rp"], dtype=float)

    slope, intercept, log10_sig = pc_result.theta
    bp_rp_int = slope * (log_p - mean_log_p) + intercept

    e_bprp = bp_rp_obs - bp_rp_int
    out["E_bprp"] = e_bprp
    out["A_G_calc"] = r_g * e_bprp

    # Propagated uncertainty: color noise + coefficient uncertainty + intrinsic scatter
    snr_bp = np.array(data["phot_bp_mean_flux_over_error"], dtype=float)
    snr_rp = np.array(data["phot_rp_mean_flux_over_error"], dtype=float)
    sigma_color = (2.5 / np.log(10)) * np.sqrt(1 / snr_bp**2 + 1 / snr_rp**2)
    sigma_intrinsic = 10.0**log10_sig
    out["sigma_E"] = np.sqrt(sigma_color**2 + sigma_intrinsic**2)

    return out


def build_quality_mask(data):
    """Build a quality mask for reddening analysis.

    Requires finite E, BP/RP SNR above threshold, BP/RP excess within
    the Gaia envelope, sigma_E below cutoff, and physical E bounds.
    """
    e = np.array(data["E_bprp"], dtype=float)
    sigma_e = np.array(data["sigma_E"], dtype=float)
    bp_rp = np.array(data["bp_rp"], dtype=float)
    snr_bp = np.array(data["phot_bp_mean_flux_over_error"], dtype=float)
    snr_rp = np.array(data["phot_rp_mean_flux_over_error"], dtype=float)
    excess = np.array(data["phot_bp_rp_excess_factor"], dtype=float)

    finite = np.isfinite(e) & np.isfinite(sigma_e) & np.isfinite(bp_rp)
    bp_ok = snr_bp > MIN_BP_SNR
    rp_ok = snr_rp > MIN_RP_SNR
    excess_ok = (excess > 1.0 + 0.015 * bp_rp**2) & (excess < 1.3 + 0.06 * bp_rp**2)
    sigma_ok = sigma_e <= MAX_SIGMA_E
    physical = (e >= MIN_EBPRP) & (e <= MAX_EBPRP)

    components = {
        "finite": finite,
        "bp_snr": bp_ok,
        "rp_snr": rp_ok,
        "excess": excess_ok,
        "sigma_e": sigma_ok,
        "physical": physical,
    }
    adopted = finite & bp_ok & rp_ok & excess_ok & sigma_ok & physical
    return adopted, components

In [ ]:
data = np.load(Path("rrlyrae_calibration_sample.npz"), allow_pickle=True)
rrlyrae = table.Table({k: data[k] for k in data.files})

rrab_cal = rrlyrae[rrlyrae["best_classification"] == "RRab"]
rrc_cal = rrlyrae[rrlyrae["best_classification"] == "RRc"]

rrab_pc_result = nuts_sample(LinearGaussianLikelihood(*build_pc_arrays(rrab_cal)))
rrc_pc_result = nuts_sample(LinearGaussianLikelihood(*build_pc_arrays(rrc_cal)))

rrab_mean_log_p = float(np.mean(np.log10(rrab_cal["rrlyrae_representative_period"])))
rrc_mean_log_p = float(np.mean(np.log10(rrc_cal["rrlyrae_representative_period"])))

print(f"RRab PC fit: slope={rrab_pc_result.theta[0]:.4f}, "
      f"intercept={rrab_pc_result.theta[1]:.4f}")
print(f"RRc  PC fit: slope={rrc_pc_result.theta[0]:.4f}, "
      f"intercept={rrc_pc_result.theta[1]:.4f}")

In [ ]:
query = """
SELECT *
FROM gaiadr3.vari_rrlyrae AS vr
JOIN gaiadr3.gaia_source AS gs
    ON vr.source_id = gs.source_id
"""

full_catalog = GaiaData(query)
full_data = full_catalog.data

rrab_full = full_data[full_data["best_classification"] == "RRab"]
rrc_full = full_data[full_data["best_classification"] == "RRc"]
print(f"Full catalog: {len(full_data):,} stars "
      f"(RRab: {len(rrab_full):,}, RRc: {len(rrc_full):,})")

In [ ]:
rrab_ext = compute_extinction(rrab_full, rrab_pc_result, rrab_mean_log_p)
rrc_ext = compute_extinction(rrc_full, rrc_pc_result, rrc_mean_log_p)
rrlyrae_extinction = table.vstack([rrab_ext, rrc_ext])

adopted_mask, components = build_quality_mask(rrlyrae_extinction)
removed_mask = components["finite"] & ~adopted_mask

print(f"Finite E(BP-RP):  {components['finite'].sum():,}")
print(f"Adopted (clean):  {adopted_mask.sum():,}")
print(f"Removed:          {removed_mask.sum():,}")

In [ ]:
e_finite = np.array(rrlyrae_extinction["E_bprp"], dtype=float)
finite = components["finite"]
vmin = float(min(0.0, np.nanpercentile(e_finite[finite], 0.5)))
vmax = float(np.nanpercentile(e_finite[finite], 99.5))

fig, ax = plot_aitoff_reddening(
    np.array(rrlyrae_extinction["l"], dtype=float)[finite],
    np.array(rrlyrae_extinction["b"], dtype=float)[finite],
    e_finite[finite],
    vmin=vmin, vmax=vmax,
)
ax.set_title("Uncut RR Lyrae reddening map", pad=20)
plt.show()

In [ ]:
finite = components["finite"]
axes = plot_reddening_quality_diagnostics(
    np.array(rrlyrae_extinction["bp_rp"], dtype=float)[finite],
    np.array(rrlyrae_extinction["phot_bp_rp_excess_factor"], dtype=float)[finite],
    np.array(rrlyrae_extinction["sigma_E"], dtype=float)[finite],
    max_sigma_e=MAX_SIGMA_E,
)
plt.show()

In [ ]:
fig, ax = plot_aitoff_reddening(
    np.array(rrlyrae_extinction["l"], dtype=float)[adopted_mask],
    np.array(rrlyrae_extinction["b"], dtype=float)[adopted_mask],
    e_finite[adopted_mask],
    vmin=vmin, vmax=vmax,
)
ax.set_title(f"Quality-cut RR Lyrae reddening map ($N$ = {adopted_mask.sum():,})", pad=20)
plt.show()

In [ ]:
from dustmaps.config import config as dustmaps_config
import dustmaps.sfd
from dustmaps.sfd import SFDQuery
from astropy.coordinates import SkyCoord
import astropy.units as u

DUSTMAPS_DATA_DIR = Path(".dustmaps-data")
DUSTMAPS_DATA_DIR.mkdir(parents=True, exist_ok=True)
dustmaps_config["data_dir"] = str(DUSTMAPS_DATA_DIR.resolve())

try:
    sfd = SFDQuery()
except FileNotFoundError:
    dustmaps.sfd.fetch()
    sfd = SFDQuery()

rrlyrae_clean = rrlyrae_extinction[adopted_mask].copy()
coords = SkyCoord(
    l=np.array(rrlyrae_clean["l"], dtype=float) * u.deg,
    b=np.array(rrlyrae_clean["b"], dtype=float) * u.deg,
    frame="galactic",
)
rrlyrae_clean["sfd_ebv"] = sfd(coords)

empirical = np.array(rrlyrae_clean["E_bprp"], dtype=float)
sfd_ebv = np.array(rrlyrae_clean["sfd_ebv"], dtype=float)

print(f"Cleaned sample with SFD: {len(rrlyrae_clean):,} stars")
print(f"Median SFD E(B-V):       {np.nanmedian(sfd_ebv):.4f} mag")
print(f"Median empirical E:      {np.nanmedian(empirical):.4f} mag")

In [ ]:
fig, ax = plot_sfd_comparison(sfd_ebv, empirical)
plt.show()

In [ ]:
SIMILAR_SCALE_MAX = 2.0
LARGE_SFD_MIN = 10.0

finite_mask = np.isfinite(sfd_ebv) & np.isfinite(empirical)
similar_mask = finite_mask & (sfd_ebv <= SIMILAR_SCALE_MAX)
large_mask = finite_mask & (sfd_ebv > LARGE_SFD_MIN)

x_sim = sfd_ebv[similar_mask]
y_sim = empirical[similar_mask]
slope, intercept = np.polyfit(x_sim, y_sim, 1)
r_sim, _ = pearsonr(x_sim, y_sim)

print(f"Similar-scale: {similar_mask.sum():,} stars, slope={slope:.3f}, R²={r_sim**2:.3f}")
print(f"Large-SFD:     {large_mask.sum():,} stars")

axes = plot_sfd_regime_decomposition(
    sfd_ebv, empirical, similar_mask, large_mask,
    slope, intercept, r_sim**2,
    similar_scale_max=SIMILAR_SCALE_MAX, large_sfd_min=LARGE_SFD_MIN,
)
plt.show()